# Run the world-camera transformation pipeline

Point this at a directory of raw sensor chunks, give it somewhere to write, and
it runs Geoff's reconstruction pipeline over every world chunk.

The work is done by `sensors_utility.process_raw_recording`, which is the
project's standard entry point for turning a raw recording into processed
chunks. It calls `world_util.world_transformation_pipeline`, the Python
equivalent of MATLAB `reconstructionPipeline.m`, which takes raw sensor counts
through six stages:

1. convert to double
2. linearize the sensor response, marking saturated pixels `Inf`
3. impute the ceiling and floor pixels
4. flat-field correction
5. equalize the RGB channels
6. convert to absolute radiance

The result is a **Bayer** radiance map, exactly as MATLAB returns. Call
`world_util.demosaic_radiance_map_rcd` separately when a full RGB image is
wanted; the last section of this notebook shows that.

`process_raw_recording` also processes the minispectrometer chunks when the
recording has them, writing those beside the world output. If the recording has
none, that pass is a harmless no-op.

Note this is a different path from `preprocessing_pipeline.generate_world_videos`,
which builds viewable `W.avi` files using individual stage flags. This notebook
produces calibrated radiance, not video.

> **Kernel.** This needs an environment with `hdf5storage`, which the default
> `python3` (3.13) kernel does not have. Use the **myenv** kernel
> (`/opt/anaconda3/envs/pylids`, Python 3.10) — the same 3.10 environment the
> rest of the preprocessing code runs under. The notebook is already set to it;
> if you see `ModuleNotFoundError: No module named 'hdf5storage'`, the kernel
> has been switched back to the default.

## Set the input and output paths

These two paths are the only things you should need to change.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt


# ---------------------------------------------------------------------------
# INPUT: directory holding the raw chunks. These are the .npy files a recording
# writes out, named world_<n>.npy alongside world_<n>_metadata.npy. For a FLIC
# recording this is the GKA/<recording number> directory.
RAW_RECORDING_PATH: Path = Path("/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/<subject>/<activity>/GKA/1")

# OUTPUT: directory to write into. process_raw_recording creates a W
# subdirectory for the world camera, holding one MATLAB v7.3 file per chunk
# named world_chunk<n>.mat, and an M subdirectory for the minispectrometer.
OUTPUT_PATH: Path = Path("/Volumes/FLIC_processing/world_radiance_scratch")

# Set True to regenerate chunks that have already been written.
OVERWRITE_EXISTING: bool = False

# Limit the work to a slice of chunks per sensor, as (start, end). Use
# (0, None) for all, or something like (0, 2) for a quick trial run before
# committing to a whole recording, since each frame takes about 1.5 seconds.
CHUNK_RANGES: dict[str, tuple[int, int | None]] = {"W": (0, None), "M": (0, None)}
# ---------------------------------------------------------------------------


# Make world_util and sensors_utility importable.
PROJECT_ROOT: Path = Path("/Users/zacharykelly/Documents/MATLAB/projects/lightLoggerAnalysis")
SENSOR_UTILITY_DIR: Path = PROJECT_ROOT / "code" / "library" / "sensor_utility"
sys.path.append(str(SENSOR_UTILITY_DIR))

import world_util
import sensors_utility

# The world chunks land here once the pipeline has run.
WORLD_OUTPUT_PATH: Path = OUTPUT_PATH / "W"

print(f"Reading from : {RAW_RECORDING_PATH}")
print(f"Writing to   : {OUTPUT_PATH}")

## Find the raw world chunks

This reuses the same file-pairing logic the production code uses, so the chunks are picked up and ordered exactly as they are during a normal run.

In [ ]:
# process_raw_recording finds the chunks itself. This cell only reports what it
# is going to see, so a wrong path is caught before a long run starts.
if not RAW_RECORDING_PATH.is_dir():
    raise FileNotFoundError(
        f"{RAW_RECORDING_PATH} is not a directory. Set RAW_RECORDING_PATH above "
        f"to a folder of raw sensor chunks."
    )

# group_sensors_files pairs each *_metadata.npy with its matching data file and
# returns them in natural chunk order, keyed by sensor letter. "W" is the world
# camera and "M" the minispectrometer.
chunk_files: dict[str, list[tuple[str, str]]] = sensors_utility.group_sensors_files(
    str(RAW_RECORDING_PATH)
)

if not chunk_files["W"]:
    raise FileNotFoundError(
        f"No world chunks found in {RAW_RECORDING_PATH}. Expected files named "
        f"world_<n>.npy alongside world_<n>_metadata.npy."
    )

print(f"World chunks : {len(chunk_files['W'])}")
print(f"MS chunks    : {len(chunk_files['M'])}")
print()
for chunk_index, (metadata_path, data_path) in enumerate(chunk_files["W"][:5]):
    frame_count: int = len(np.load(data_path, mmap_mode="r"))
    print(f"  world chunk {chunk_index}: {Path(data_path).name}  ({frame_count} frames)")
if len(chunk_files["W"]) > 5:
    print(f"  ... and {len(chunk_files['W']) - 5} more")

## Run the pipeline

`process_raw_recording` reads each chunk, passes the world frames through
`world_transformation_pipeline`, and writes the result as a MATLAB v7.3 file
containing a `data` array of radiance maps and a `metadata` struct with the AGC
settings and timestamps.

Expect roughly 1.5 seconds per frame. Most of that is the triangulation inside
the imputation stage, so a chunk of 100 frames takes a couple of minutes.

In [ ]:
sensors_utility.process_raw_recording(
    str(RAW_RECORDING_PATH),
    str(OUTPUT_PATH),
    overwrite_existing=OVERWRITE_EXISTING,
    verbose=True,
    chunk_ranges=CHUNK_RANGES,
)

written: list[Path] = sorted(WORLD_OUTPUT_PATH.glob("world_chunk*.mat"))
print(f"\nWrote {len(written)} world chunk file(s) to {WORLD_OUTPUT_PATH}")
for path in written[:5]:
    print(f"  {path.name}  ({path.stat().st_size / 1e6:.1f} MB)")

## Check one result

Load a chunk back and confirm it looks like calibrated radiance.

In [ ]:
if not written:
    raise RuntimeError("No output files were written, so there is nothing to check.")

# hdf5storage writes MATLAB v7.3, which is HDF5, so h5py reads it directly.
import h5py

with h5py.File(written[0], "r") as processed:
    # MATLAB v7.3 stores arrays transposed relative to NumPy, so read and undo it.
    radiance_buffer: np.ndarray = np.array(processed["data"]).T
    timestamps: np.ndarray = np.array(processed["metadata"]["timestamps"]).reshape(-1)

print(f"radiance buffer shape : {radiance_buffer.shape}   (frames, rows, cols)")
print(f"dtype                 : {radiance_buffer.dtype}")
print(f"timestamps            : {timestamps.size}")

# The pipeline imputes saturated pixels, so a correct result has no Inf left in
# it. Any NaN would mean a chunk had nothing valid to interpolate from.
finite_values: np.ndarray = radiance_buffer[np.isfinite(radiance_buffer)]
print(f"\nnon-finite pixels     : {radiance_buffer.size - finite_values.size}")
print(f"radiance min / median / max : "
      f"{finite_values.min():.4g} / {np.median(finite_values):.4g} / {finite_values.max():.4g}")

## Look at a frame

The pipeline returns a Bayer radiance map, so the raw result still carries the
mosaic. `demosaic_radiance_map_rcd` turns it into an RGB radiance image, which
is the same thing MATLAB `demosaicRadianceMapRCD` produces.

In [ ]:
frame_index: int = 0
bayer_radiance: np.ndarray = radiance_buffer[frame_index]

# Ratio-corrected demosaicing, returning (rows, cols, 3).
rgb_radiance: np.ndarray = world_util.demosaic_radiance_map_rcd(
    bayer_radiance, bayer_pattern="BGGR"
)

# Radiance is unbounded, so stretch the middle 98% into the display range
# rather than letting one bright pixel flatten everything else.
def stretch_for_display(values: np.ndarray) -> np.ndarray:
    """Scale finite values into 0-1 using their 1st and 99th percentiles."""
    finite: np.ndarray = values[np.isfinite(values)]
    if finite.size == 0:
        return np.zeros_like(values)
    lower, upper = np.percentile(finite, (1, 99))
    if upper <= lower:
        return np.zeros_like(values)
    return np.clip((values - lower) / (upper - lower), 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
fig.suptitle(f"Frame {frame_index}", fontsize=15, fontweight="bold")

axes[0].imshow(stretch_for_display(bayer_radiance), cmap="gray")
axes[0].set_title("Bayer radiance (pipeline output)")

axes[1].imshow(stretch_for_display(rgb_radiance))
axes[1].set_title("RGB radiance (after RCD demosaicing)")

for axis in axes:
    axis.axis("off")

plt.show()